# Dynamic scripting: C#, Python and JavaScript
**EN** — Let analysts filter and query live data with scripts compiled at runtime.

**ID** — Analis dapat memfilter dan melakukan query data langsung dengan skrip yang dikompilasi saat runtime.

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
// Using a local build instead of nuget.org? Uncomment and point to <repo>/artifacts/nuget (absolute path):
// #i "nuget: C:/src/bigpipe/artifacts/nuget"
#r "nuget: BigPipe.Analytics.Scripting, 0.1.0"
using BigPipe.Client;
using BigPipe.Client.Admin;
using BigPipe.Analytics;
var bootstrap = Environment.GetEnvironmentVariable("BIGPIPE_BOOTSTRAP") ?? "localhost:9092";
var admin = new BigPipeAdminClient(Environment.GetEnvironmentVariable("BIGPIPE_ADMIN") ?? "http://localhost:9644");
var http = new BigPipeHttpClient(Environment.GetEnvironmentVariable("BIGPIPE_HTTP") ?? "http://localhost:8082");
string Unique(string p) => $"nb-{p}-{Guid.NewGuid().ToString("N")[..6]}";
var cluster = await admin.GetClusterAsync();
Console.WriteLine($"Connected to {cluster.ClusterId} (BigPipe {cluster.Version}) — {cluster.Credit}");

In [ ]:
using BigPipe.Analytics.Scripting;
var topic = Unique("orders");
await admin.CreateTopicAsync(topic, 3);
var rng = new Random(3);
var channels = new[] { "app", "web", "pos" };
await http.ProduceAsync(topic, Enumerable.Range(0, 600).Select(i => new HttpProduceRecord
{
    Key = $"order-{i}",
    Value = new { total = rng.Next(10, 900) * 1000, items = rng.Next(1, 8), channel = channels[i % 3] },
    Headers = new Dictionary<string, string> { ["region"] = i % 4 == 0 ? "SG" : "ID" },
}));
var window = await BigPipeSource.Kafka(bootstrap, topic, from: OffsetReset.Earliest).CollectAsync(600);
Console.WriteLine($"{window.Count} orders in the window");

In an application, `ScriptEngines.Create(ScriptLanguage.CSharp)` compiles C# filters with Roslyn. Inside this notebook C# *is* the kernel language, so the C# version is simply a lambda; Python and JavaScript use the scripting engines.

Di aplikasi, `ScriptEngines.Create(ScriptLanguage.CSharp)` mengompilasi filter C# dengan Roslyn. Di notebook ini C# sudah menjadi bahasa kernel, jadi versi C#-nya cukup berupa lambda; Python dan JavaScript memakai mesin skrip.

In [ ]:
Func<AnalyticsRecord, bool> csharpFilter = r => r.Num("total") > 500000 && r.Header("region") == "ID";
var csMatches = window.Where(csharpFilter).ToList();
var results = new List<object> { new { language = "C# (lambda)", matches = csMatches.Count, avgItems = Math.Round(csMatches.Average(x => x.Num("items")), 3) } };
foreach (var (lang, filter, query) in new[]
{
    (ScriptLanguage.Python, "r.num('total') > 500000 and r.header('region') == 'ID'", "result = sum(x.num('items') for x in records) / len(records)"),
    (ScriptLanguage.JavaScript, "r.num('total') > 500000 && r.header('region') === 'ID'", "records.reduce((s, x) => s + x.num('items'), 0) / records.length"),
})
{
    using var engine = ScriptEngines.Create(lang);
    var matches = window.Where(engine.CompileFilter(filter)).ToList();
    results.Add(new { language = lang.ToString(), matches = matches.Count, avgItems = Math.Round(Convert.ToDouble(engine.Query(query, matches)), 3) });
}
results

Server-side, the broker itself evaluates **bpql** filters — the same idea without any client code:

`GET /v1/topics/{topic}/stream?filter=this.total > 500000 && header("region") == "ID"`